# Huberman Lab Podcast

### Setup

In [7]:
import os
import pandas as pd
import pickle
from youtube_transcript_api import YouTubeTranscriptApi

In [8]:
ytt_api = YouTubeTranscriptApi()

In [9]:
data = pd.read_csv('data/huberman_videos.csv', encoding='latin1', index_col=False)
data.head()

,id,url,title,video_key
0,0,https://www.youtube.com/watch?v=4b6bwcWK6GE&li...,Welcome to the Huberman Lab Podcast,4b6bwcWK6GE
1,1,https://www.youtube.com/watch?v=H-XfCl-HpRM&li...,How Your Brain Works & Changes,H-XfCl-HpRM
2,2,https://www.youtube.com/watch?v=nm1TxQj9IsQ&li...,Master Your Sleep & Be More Alert When Awake,nm1TxQj9IsQ
3,3,https://www.youtube.com/watch?v=nwSkFq4tyC0&li...,"Using Science to Optimize Sleep, Learning & Me...",nwSkFq4tyC0
4,4,https://www.youtube.com/watch?v=NAATB55oxeQ&li...,"How to Defeat Jet Lag, Shift Work & Sleeplessness",NAATB55oxeQ


### Prepare a utility dict

This will be beneficial when creating the embeddings and allow us to pass the title of each video_id into the embedding module 

In [10]:
title_dict = data.set_index('video_key')['title'].to_dict()
print(f'Video title at index zero:\n{title_dict[data.video_key[:2][0]]}\n')
print(f'Video title at index one:\n{title_dict[data.video_key[:2][1]]}')

Video title at index zero:
Welcome to the Huberman Lab Podcast

Video title at index one:
How Your Brain Works & Changes


In [11]:
with open('data/title_dict.pkl', 'wb') as f:
    pickle.dump(title_dict, f)

### Generate and Store Transcripts

TODO: Update this to utilize `Mongo` opposed to it's current `.txt` storage

In [12]:
import time
import random
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

os.makedirs('data/documents', exist_ok=True)

def get_transcript_with_retry(video_id, max_retries=3, base_delay=1):
    """
    Fetch transcript with exponential backoff retry logic
    """
    for attempt in range(max_retries):
        try:
            # Use list_transcripts instead of deprecated get_transcript
            transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
            
            # Try to find English transcript (auto-generated or manual)
            try:
                transcript = transcript_list.find_transcript(['en'])
            except NoTranscriptFound:
                # If no English, try auto-generated English
                try:
                    transcript = transcript_list.find_generated_transcript(['en'])
                except NoTranscriptFound:
                    # If still no English, get the first available transcript
                    transcript = transcript_list._transcript_list[0] if transcript_list._transcript_list else None
                    if transcript is None:
                        raise NoTranscriptFound(video_id, [], None)
            
            # Fetch the actual transcript data
            return transcript.fetch()
            
        except (TranscriptsDisabled, VideoUnavailable) as e:
            # These errors won't be resolved by retrying
            raise e
        except Exception as e:
            if attempt == max_retries - 1:
                # Last attempt failed
                raise e
            
            # Calculate delay with exponential backoff and jitter
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"Attempt {attempt + 1} failed for video {video_id}: {str(e)}. Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
    
    return None

# Process videos with improved error handling
successful_downloads = 0
failed_downloads = 0
skipped_existing = 0

for idx, (video_key, title) in enumerate(zip(data['video_key'], data['title'])):
    filename = f'data/documents/{video_key}.txt'
    
    # Skip if file already exists and has content
    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        print(f"Skipping {video_key} - file already exists")
        skipped_existing += 1
        continue
    
    print(f"Processing video {idx + 1}/{len(data)}: {video_key} - {title[:50]}...")
    
    try:
        # Get transcript with retry logic
        fetched_transcript = get_transcript_with_retry(video_key)
        
        # Write transcript to file
        with open(filename, 'w', encoding='utf-8') as file:
            for snippet in fetched_transcript:
                file.write(snippet['text'] + '\n')
        
        successful_downloads += 1
        print(f"✓ Successfully downloaded transcript for {video_key}")
        
        # Add small delay between requests to be respectful
        time.sleep(0.5)
        
    except TranscriptsDisabled:
        print(f"✗ Transcripts disabled for video {video_key}")
        failed_downloads += 1
    except VideoUnavailable:
        print(f"✗ Video {video_key} is unavailable")
        failed_downloads += 1
    except NoTranscriptFound:
        print(f"✗ No transcript found for video {video_key}")
        failed_downloads += 1
    except Exception as e:
        print(f"✗ Error processing video {video_key}: {str(e)}")
        failed_downloads += 1

print(f"\n=== Summary ===")
print(f"Successful downloads: {successful_downloads}")
print(f"Failed downloads: {failed_downloads}")
print(f"Skipped (already exist): {skipped_existing}")
print(f"Total processed: {successful_downloads + failed_downloads + skipped_existing}")
print(f"Success rate: {(successful_downloads / max(1, successful_downloads + failed_downloads)) * 100:.1f}%")

Skipping 4b6bwcWK6GE - file already exists
Processing video 2/292: H-XfCl-HpRM - How Your Brain Works & Changes...
Attempt 1 failed for video H-XfCl-HpRM: 'TranscriptList' object has no attribute '_transcript_list'. Retrying in 1.46 seconds...
Attempt 2 failed for video H-XfCl-HpRM: 'TranscriptList' object has no attribute '_transcript_list'. Retrying in 2.16 seconds...
✗ Error processing video H-XfCl-HpRM: 'TranscriptList' object has no attribute '_transcript_list'
Skipping nm1TxQj9IsQ - file already exists
Processing video 4/292: nwSkFq4tyC0 - Using Science to Optimize Sleep, Learning & Metabo...
Attempt 1 failed for video nwSkFq4tyC0: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=nwSkFq4tyC0! This is most likely caused by:

Request to YouTube failed: 429 Client Error: Too Many Requests for url: https://www.youtube.com/api/timedtext?v=nwSkFq4tyC0&ei=TUiNaMm7EeCTsfIPhIaz0Q8&caps=asr&opi=112496729&exp=xpe&xoaf=4&hl=en&ip=0.0.0.0&ipbits=0&expire=17541147

KeyboardInterrupt: 